**Step 1: Installing and importing libraries**

In [ ]:
!pip -q install imbalanced-learn

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, average_precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN


**Step 2: Basic data inspection and imbalance check**

In [ ]:
df = pd.read_csv("creditcard.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist()[:10], "...")

print("\nClass counts:")
print(df["Class"].value_counts())

print("\nClass percent:")
print((df["Class"].value_counts(normalize=True) * 100).round(4))


Shape: (284807, 31)
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9'] ...

Class counts:
Class
0    284315
1       492
Name: count, dtype: int64

Class percent:
Class
0    99.8273
1     0.1727
Name: proportion, dtype: float64


**Step 3: Train–test split**

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train class %:")
print((y_train.value_counts(normalize=True) * 100).round(4))
print("\nTest class %:")
print((y_test.value_counts(normalize=True) * 100).round(4))


Train class %:
Class
0    99.8271
1     0.1729
Name: proportion, dtype: float64

Test class %:
Class
0    99.828
1     0.172
Name: proportion, dtype: float64


**Step 4: Evaluating the model**

In [ ]:
def evaluate_model(name, model, X_test, y_test, threshold=0.5):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)

    print("=" * 70)
    print(name)
    print("-" * 70)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, pred))
    print("\nClassification Report:")
    print(classification_report(y_test, pred, digits=4))
    print("ROC-AUC:", roc_auc_score(y_test, proba))
    print("PR-AUC :", average_precision_score(y_test, proba))


**Step 5: Defining the samplers and models**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN

sampler_factories = {
    "RandomOverSampler": lambda: RandomOverSampler(random_state=42),
    "SMOTE": lambda: SMOTE(random_state=42),
    "ADASYN": lambda: ADASYN(random_state=42),
}

model_factories = {
    "LogisticRegression": lambda: LogisticRegression(max_iter=5000),
    "RandomForest": lambda: RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
    "DecisionTree": lambda: DecisionTreeClassifier(random_state=42),
}

**Step 6: Training  the model with logistic Regression with samplers like ROS, SMOTE, ADASYN**

In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

lr_results = {}

for sampler_name, sampler_fn in sampler_factories.items():
    pipe = Pipeline(steps=[
        ("sampler", sampler_fn()),
        ("scaler", StandardScaler()),
        ("model", model_factories["LogisticRegression"]())
    ])

    pipe.fit(X_train, y_train)
    evaluate_model(f"Logistic Regression + {sampler_name}", pipe, X_test, y_test, threshold=0.5)
    lr_results[sampler_name] = pipe

Logistic Regression + RandomOverSampler
----------------------------------------------------------------------
Confusion Matrix:
[[55481  1383]
 [    8    90]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9999    0.9757    0.9876     56864
           1     0.0611    0.9184    0.1146        98

    accuracy                         0.9756     56962
   macro avg     0.5305    0.9470    0.5511     56962
weighted avg     0.9982    0.9756    0.9861     56962

ROC-AUC: 0.9719515162564745
PR-AUC : 0.711174179613322
Logistic Regression + SMOTE
----------------------------------------------------------------------
Confusion Matrix:
[[56296   568]
 [   10    88]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9998    0.9900    0.9949     56864
           1     0.1341    0.8980    0.2334        98

    accuracy                         0.9899     56962
   macro avg     0.5670    0.9440    0.6142 

**Step 7: Training the model with Random Forest with samplers like ROS, SMOTE, ADASYN**

In [ ]:
rf_results = {}

for sampler_name, sampler_fn in sampler_factories.items():
    pipe = Pipeline(steps=[
        ("sampler", sampler_fn()),
        ("model", model_factories["RandomForest"]())
    ])

    pipe.fit(X_train, y_train)
    evaluate_model(f"Random Forest + {sampler_name}", pipe, X_test, y_test, threshold=0.5)
    rf_results[sampler_name] = pipe

Random Forest + RandomOverSampler
----------------------------------------------------------------------
Confusion Matrix:
[[56860     4]
 [   21    77]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9996    0.9999    0.9998     56864
           1     0.9506    0.7857    0.8603        98

    accuracy                         0.9996     56962
   macro avg     0.9751    0.8928    0.9301     56962
weighted avg     0.9995    0.9996    0.9995     56962

ROC-AUC: 0.9612142971989021
PR-AUC : 0.8672116271013841
Random Forest + SMOTE
----------------------------------------------------------------------
Confusion Matrix:
[[56846    18]
 [   17    81]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9997    0.9997    0.9997     56864
           1     0.8182    0.8265    0.8223        98

    accuracy                         0.9994     56962
   macro avg     0.9089    0.9131    0.9110     56962
w

**Step 8: Training the model with Decision Tree with samplers like ROS, SMOTE, ADASYN**

In [ ]:
dt_results = {}

for sampler_name, sampler_fn in sampler_factories.items():
    pipe = Pipeline(steps=[
        ("sampler", sampler_fn()),
        ("model", model_factories["DecisionTree"]())
    ])

    pipe.fit(X_train, y_train)
    evaluate_model(f"Decision Tree + {sampler_name}", pipe, X_test, y_test, threshold=0.5)
    dt_results[sampler_name] = pipe

Decision Tree + RandomOverSampler
----------------------------------------------------------------------
Confusion Matrix:
[[56836    28]
 [   29    69]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9995    0.9995    0.9995     56864
           1     0.7113    0.7041    0.7077        98

    accuracy                         0.9990     56962
   macro avg     0.8554    0.8518    0.8536     56962
weighted avg     0.9990    0.9990    0.9990     56962

ROC-AUC: 0.8517946148633905
PR-AUC : 0.5013506850802821
Decision Tree + SMOTE
----------------------------------------------------------------------
Confusion Matrix:
[[56758   106]
 [   20    78]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9996    0.9981    0.9989     56864
           1     0.4239    0.7959    0.5532        98

    accuracy                         0.9978     56962
   macro avg     0.7118    0.8970    0.7760     56962
w

**Step 9:Comparison Table for all 9 models (0.5)**

In [ ]:
import pandas as pd
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

THRESHOLD = 0.5
rows = []

def add_rows(model_name, results_dict):
    for sampler_name, pipe in results_dict.items():
        proba = pipe.predict_proba(X_test)[:, 1]
        pred = (proba >= THRESHOLD).astype(int)

        rows.append({
            "Model": model_name,
            "Sampler": sampler_name,
            "Threshold": THRESHOLD,
            "ROC-AUC": roc_auc_score(y_test, proba),
            "PR-AUC": average_precision_score(y_test, proba),
            "Precision (fraud=1)": precision_score(y_test, pred, zero_division=0),
            "Recall (fraud=1)": recall_score(y_test, pred, zero_division=0),
            "F1 (fraud=1)": f1_score(y_test, pred, zero_division=0),
            "Predicted Fraud Count": int(pred.sum())
        })

add_rows("LogisticRegression", lr_results)
add_rows("RandomForest", rf_results)
add_rows("DecisionTree", dt_results)

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values(
    by=["PR-AUC", "ROC-AUC"],
    ascending=False
).reset_index(drop=True)

print("=== RESULTS (sorted by PR-AUC, then ROC-AUC) ===")
display(results_df.round(6))

=== RESULTS (sorted by PR-AUC, then ROC-AUC) ===


,Model,Sampler,Threshold,ROC-AUC,PR-AUC,Precision (fraud=1),Recall (fraud=1),F1 (fraud=1),Predicted Fraud Count
0,RandomForest,SMOTE,0.5,0.981668,0.876012,0.818182,0.826531,0.822335,99
1,RandomForest,ADASYN,0.5,0.980948,0.874285,0.822917,0.806122,0.814433,96
2,RandomForest,RandomOverSampler,0.5,0.961214,0.867212,0.950617,0.785714,0.860335,81
3,LogisticRegression,ADASYN,0.5,0.977155,0.734268,0.128843,0.897959,0.225352,683
4,LogisticRegression,SMOTE,0.5,0.976482,0.734231,0.134146,0.897959,0.233422,656
5,LogisticRegression,RandomOverSampler,0.5,0.971952,0.711174,0.061100,0.918367,0.114577,1473
6,DecisionTree,RandomOverSampler,0.5,0.851795,0.501351,0.711340,0.704082,0.707692,97
7,DecisionTree,ADASYN,0.5,0.902138,0.346440,0.429348,0.806122,0.560284,184
8,DecisionTree,SMOTE,0.5,0.897027,0.337751,0.423913,0.795918,0.553191,184


**Step 10: Hyperparameter Tuning and Comparison table**

In [ ]:
# Step 10a: Settings & Param Grids
from sklearn.model_selection import RandomizedSearchCV

N_ITER   = 2
CV_FOLDS = 2
SCORING  = "average_precision"
RANDOM   = 42

LR_GRID = {
    "sampler__sampling_strategy": [0.1, 1.0],
    "model__C":                   [0.01, 1, 10],
    "model__penalty":             ["l1", "l2"],
    "model__solver":              ["saga"],
    "model__class_weight":        [None, "balanced"],
}
RF_GRID = {
    "sampler__sampling_strategy": [0.1, 1.0],
    "model__n_estimators":        [50, 100],
    "model__max_depth":           [10, 20],
    "model__max_features":        ["sqrt"],
    "model__class_weight":        [None, "balanced"],
}
DT_GRID = {
    "sampler__sampling_strategy": [0.1, 1.0],
    "model__max_depth":           [None, 10],
    "model__min_samples_leaf":    [1, 4],
    "model__criterion":           ["gini", "entropy"],
    "model__class_weight":        [None, "balanced"],
}
SMOTE_EXTRA  = {"sampler__k_neighbors": [3, 5]}
ADASYN_EXTRA = {"sampler__n_neighbors": [3, 5]}

In [ ]:
# Step 10b: Build Pipelines & Run Tuning
configs = [
    ("LogisticRegression", "RandomOverSampler",
     Pipeline([("sampler", RandomOverSampler(random_state=RANDOM)),
               ("scaler",  StandardScaler()),
               ("model",   LogisticRegression(max_iter=1000))]), LR_GRID),

    ("LogisticRegression", "SMOTE",
     Pipeline([("sampler", SMOTE(random_state=RANDOM)),
               ("scaler",  StandardScaler()),
               ("model",   LogisticRegression(max_iter=1000))]), {**LR_GRID, **SMOTE_EXTRA}),

    ("LogisticRegression", "ADASYN",
     Pipeline([("sampler", ADASYN(random_state=RANDOM)),
               ("scaler",  StandardScaler()),
               ("model",   LogisticRegression(max_iter=1000))]), {**LR_GRID, **ADASYN_EXTRA}),

    ("RandomForest", "RandomOverSampler",
     Pipeline([("sampler", RandomOverSampler(random_state=RANDOM)),
               ("model",   RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=RANDOM))]), RF_GRID),

    ("RandomForest", "SMOTE",
     Pipeline([("sampler", SMOTE(random_state=RANDOM)),
               ("model",   RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=RANDOM))]), {**RF_GRID, **SMOTE_EXTRA}),

    ("RandomForest", "ADASYN",
     Pipeline([("sampler", ADASYN(random_state=RANDOM)),
               ("model",   RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=RANDOM))]), {**RF_GRID, **ADASYN_EXTRA}),

    ("DecisionTree", "RandomOverSampler",
     Pipeline([("sampler", RandomOverSampler(random_state=RANDOM)),
               ("model",   DecisionTreeClassifier(random_state=RANDOM))]), DT_GRID),

    ("DecisionTree", "SMOTE",
     Pipeline([("sampler", SMOTE(random_state=RANDOM)),
               ("model",   DecisionTreeClassifier(random_state=RANDOM))]), {**DT_GRID, **SMOTE_EXTRA}),

    ("DecisionTree", "ADASYN",
     Pipeline([("sampler", ADASYN(random_state=RANDOM)),
               ("model",   DecisionTreeClassifier(random_state=RANDOM))]), {**DT_GRID, **ADASYN_EXTRA}),
]

tuned_results = {}
for model_name, sampler_name, pipe, grid in configs:
    key = f"{model_name} + {sampler_name}"
    print(f"Tuning {key} ...")
    search = RandomizedSearchCV(
        pipe, grid,
        n_iter=N_ITER, scoring=SCORING,
        cv=CV_FOLDS, refit=True,
        n_jobs=-1, random_state=RANDOM, verbose=0,
    )
    search.fit(X_train, y_train)
    tuned_results[key] = search.best_estimator_
    print(f"  Done — CV PR-AUC: {search.best_score_:.4f}")

Tuning LogisticRegression + RandomOverSampler ...
  Done — CV PR-AUC: 0.7552
Tuning LogisticRegression + SMOTE ...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


  Done — CV PR-AUC: 0.7529
Tuning LogisticRegression + ADASYN ...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


  Done — CV PR-AUC: 0.7523
Tuning RandomForest + RandomOverSampler ...
  Done — CV PR-AUC: 0.8104
Tuning RandomForest + SMOTE ...
  Done — CV PR-AUC: 0.8325
Tuning RandomForest + ADASYN ...
  Done — CV PR-AUC: 0.8173
Tuning DecisionTree + RandomOverSampler ...
  Done — CV PR-AUC: 0.6633
Tuning DecisionTree + SMOTE ...
  Done — CV PR-AUC: 0.5478
Tuning DecisionTree + ADASYN ...
  Done — CV PR-AUC: 0.4841


In [ ]:
# Step 10c: Comparison Table
THRESHOLD = 0.5
rows = []

for key, pipe in tuned_results.items():
    model_name, sampler_name = key.split(" + ")
    proba = pipe.predict_proba(X_test)[:, 1]
    pred  = (proba >= THRESHOLD).astype(int)
    rows.append({
        "Model":                 model_name,
        "Sampler":               sampler_name,
        "Threshold":             THRESHOLD,
        "ROC-AUC":               roc_auc_score(y_test, proba),
        "PR-AUC":                average_precision_score(y_test, proba),
        "Precision (fraud=1)":   precision_score(y_test, pred, zero_division=0),
        "Recall (fraud=1)":      recall_score(y_test, pred, zero_division=0),
        "F1 (fraud=1)":          f1_score(y_test, pred, zero_division=0),
        "Predicted Fraud Count": int(pred.sum()),
    })

results_df = (pd.DataFrame(rows)
                .sort_values(["PR-AUC", "ROC-AUC"], ascending=False)
                .reset_index(drop=True))

print("=== TUNED RESULTS (sorted by PR-AUC, then ROC-AUC) ===")
display(results_df.round(6))

best_row = results_df.iloc[0]
print("\n" + "=" * 70)
print("✅ BEST TUNED MODEL (by PR-AUC)")
print("=" * 70)
print(best_row)

=== TUNED RESULTS (sorted by PR-AUC, then ROC-AUC) ===


,Model,Sampler,Threshold,ROC-AUC,PR-AUC,Precision (fraud=1),Recall (fraud=1),F1 (fraud=1),Predicted Fraud Count
0,RandomForest,SMOTE,0.5,0.961338,0.869320,0.821782,0.846939,0.834171,101
1,RandomForest,RandomOverSampler,0.5,0.984447,0.862587,0.836735,0.836735,0.836735,98
2,RandomForest,ADASYN,0.5,0.975301,0.860688,0.776699,0.816327,0.796020,103
3,LogisticRegression,RandomOverSampler,0.5,0.969827,0.747860,0.412322,0.887755,0.563107,211
4,LogisticRegression,SMOTE,0.5,0.976936,0.736071,0.142857,0.908163,0.246879,623
5,LogisticRegression,ADASYN,0.5,0.976643,0.735365,0.130307,0.908163,0.227913,683
6,DecisionTree,RandomOverSampler,0.5,0.912520,0.711568,0.170886,0.826531,0.283217,474
7,DecisionTree,SMOTE,0.5,0.907452,0.623634,0.124615,0.826531,0.216578,650
8,DecisionTree,ADASYN,0.5,0.904813,0.592977,0.127389,0.816327,0.220386,628



✅ BEST TUNED MODEL (by PR-AUC)
Model                    RandomForest
Sampler                         SMOTE
Threshold                         0.5
ROC-AUC                      0.961338
PR-AUC                        0.86932
Precision (fraud=1)          0.821782
Recall (fraud=1)             0.846939
F1 (fraud=1)                 0.834171
Predicted Fraud Count             101
Name: 0, dtype: object


**Step 11 : To Save the Best Tuned Model for Streamlit Deployment**

In [ ]:
# ── Step 11: Save Best Tuned Model for Deployment ─────────────────────────────
import joblib

best_key  = f"{best_row['Model']} + {best_row['Sampler']}"
best_pipe = tuned_results[best_key]

joblib.dump(best_pipe, "best_fraud_model_tuned.pkl")

print("=" * 70)
print("✅ Model Saved Successfully")
print("=" * 70)
print(f"Model   : {best_row['Model']}")
print(f"Sampler : {best_row['Sampler']}")
print(f"PR-AUC  : {best_row['PR-AUC']}")
print(f"ROC-AUC : {best_row['ROC-AUC']}")
print(f"Saved to: best_fraud_model_tuned.pkl")

✅ Model Saved Successfully
Model   : RandomForest
Sampler : SMOTE
PR-AUC  : 0.8693198880479942
ROC-AUC : 0.9613380259954292
Saved to: best_fraud_model_tuned.pkl
